<a href="https://colab.research.google.com/github/Krishnapabbu17/Wharton-DS-Project/blob/main/notebooks/Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# WHSDC 2026 — Part 1A (Better Method)
# Logistic Regression using season-derived team features
# =========================

# --- 0) Install deps (Colab) ---
!pip -q install pandas openpyxl numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# --- 1) Load files ---
# Make sure you are in the repo folder OR update these paths.
whl_path = "data/whl_2025 (1).xlsx"
matchups_path = "data/WHSDSC_Rnd1_matchups.xlsx"

# Read WHL (xlsx/csv)
if whl_path.lower().endswith(".csv"):
    whl = pd.read_csv(whl_path)
else:
    whl = pd.read_excel(whl_path, sheet_name=0)

matchups = pd.read_excel(matchups_path, sheet_name=0)

# --- 2) Normalize column names (very important for Foundry/Excel quirks) ---
def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().lower().replace(" ", "_") for c in out.columns]
    return out

whl = normalize_cols(whl)
matchups = normalize_cols(matchups)

print("WHL shape:", whl.shape)
print("Matchups shape:", matchups.shape)

# --- 3) Find required columns in WHL ---
def pick_first_existing(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

home_team_col = pick_first_existing(whl, ["home_team", "hometeam", "home"])
away_team_col = pick_first_existing(whl, ["away_team", "awayteam", "away"])
home_goals_col = pick_first_existing(whl, ["home_goals", "homegoal", "goals_home"])
away_goals_col = pick_first_existing(whl, ["away_goals", "awaygoal", "goals_away"])
home_xg_col = pick_first_existing(whl, ["home_xg", "homexg", "xg_home"])
away_xg_col = pick_first_existing(whl, ["away_xg", "awayxg", "xg_away"])

# game identifier
gid_col = pick_first_existing(whl, ["game_id", "gameid", "id", "match_id", "matchid"])

# If no game_id, create one from date+teams if possible
date_col = pick_first_existing(whl, ["date", "game_date", "gamedate"])
if gid_col is None:
    if date_col is not None and home_team_col is not None and away_team_col is not None:
        whl["_game_id"] = (
            whl[date_col].astype(str) + "__" +
            whl[home_team_col].astype(str) + "__" +
            whl[away_team_col].astype(str)
        )
        gid_col = "_game_id"
    else:
        raise ValueError("Couldn't find a game id column OR build one (need date + home_team + away_team).")

required = [home_team_col, away_team_col, home_goals_col, away_goals_col, home_xg_col, away_xg_col, gid_col]
if any(c is None for c in required):
    raise ValueError(f"Missing one of required columns. Found: "
                     f"home_team={home_team_col}, away_team={away_team_col}, "
                     f"home_goals={home_goals_col}, away_goals={away_goals_col}, "
                     f"home_xg={home_xg_col}, away_xg={away_xg_col}, game_id={gid_col}")

print("Using columns:")
print(" game_id:", gid_col)
print(" home_team:", home_team_col, "| away_team:", away_team_col)
print(" home_goals:", home_goals_col, "| away_goals:", away_goals_col)
print(" home_xg:", home_xg_col, "| away_xg:", away_xg_col)

# --- 4) Create GAME-LEVEL table (1 row per game) ---
games = (
    whl
    .drop_duplicates(subset=[gid_col])
    [[gid_col, home_team_col, away_team_col, home_goals_col, away_goals_col, home_xg_col, away_xg_col]]
    .copy()
)

games.columns = ["game_id", "home_team", "away_team", "home_goals", "away_goals", "home_xg", "away_xg"]
print("Games table shape (should be ~ number of games):", games.shape)

# target label for training
games["home_win"] = (games["home_goals"] > games["away_goals"]).astype(int)

# --- 5) Build TEAM-GAME rows (2 rows per game: home perspective + away perspective) ---
rows = []
for _, r in games.iterrows():
    rows.append({
        "team": r["home_team"],
        "goals_for": r["home_goals"],
        "goals_against": r["away_goals"],
        "xg_for": r["home_xg"],
        "xg_against": r["away_xg"],
        "win": 1 if r["home_goals"] > r["away_goals"] else 0
    })
    rows.append({
        "team": r["away_team"],
        "goals_for": r["away_goals"],
        "goals_against": r["home_goals"],
        "xg_for": r["away_xg"],
        "xg_against": r["home_xg"],
        "win": 1 if r["away_goals"] > r["home_goals"] else 0
    })

team_games = pd.DataFrame(rows)

# --- 6) Team table = season summary per team ---
team_table = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games=("win", "count"),
        wins=("win", "sum"),
        avg_goals_for=("goals_for", "mean"),
        avg_goals_against=("goals_against", "mean"),
        avg_xg_for=("xg_for", "mean"),
        avg_xg_against=("xg_against", "mean"),
    )
)

team_table["goal_diff"] = team_table["avg_goals_for"] - team_table["avg_goals_against"]
team_table["xg_diff"] = team_table["avg_xg_for"] - team_table["avg_xg_against"]
team_table["win_pct"] = team_table["wins"] / team_table["games"]

print("Team table teams:", len(team_table))
display(team_table.head())

# --- 7) Create training features for each historical game (home minus away team season stats) ---
# Merge team season stats onto games for home and away sides
home_stats = team_table[["team", "xg_diff", "goal_diff", "win_pct"]].rename(columns={"team": "home_team"})
away_stats = team_table[["team", "xg_diff", "goal_diff", "win_pct"]].rename(columns={"team": "away_team"})

train = games.merge(home_stats, on="home_team", how="left").merge(away_stats, on="away_team", how="left", suffixes=("_home", "_away"))

# Delta features
train["xg_diff_delta"] = train["xg_diff_home"] - train["xg_diff_away"]
train["goal_diff_delta"] = train["goal_diff_home"] - train["goal_diff_away"]
train["win_pct_delta"] = train["win_pct_home"] - train["win_pct_away"]

feature_cols = ["xg_diff_delta", "goal_diff_delta", "win_pct_delta"]
X = train[feature_cols].fillna(0.0).values
y = train["home_win"].values

# --- 8) Train logistic regression (learn weights from data) ---
model = LogisticRegression(max_iter=500)
model.fit(X, y)

print("\nLearned coefficients (higher => increases home win probability):")
for name, coef in zip(feature_cols, model.coef_[0]):
    print(f"  {name:>14}: {coef:+.4f}")
print(f"  {'intercept':>14}: {model.intercept_[0]:+.4f}")

# Optional: quick CV accuracy check (just to sanity check; not required)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print("\nCV accuracy (sanity check):", acc.mean().round(4), "+/-", acc.std().round(4))

# --- 9) Predict 16 Round 1 matchup probabilities ---
# Identify home/away team column names in matchups
m_home_col = pick_first_existing(matchups, ["home_team", "hometeam", "home"])
m_away_col = pick_first_existing(matchups, ["away_team", "awayteam", "away"])

if m_home_col is None or m_away_col is None:
    raise ValueError(f"Couldn't find home_team/away_team columns in matchups. Columns: {matchups.columns.tolist()}")

# attach team stats to matchups
m = matchups.copy()
m = m.rename(columns={m_home_col: "home_team", m_away_col: "away_team"})

m = m.merge(home_stats, on="home_team", how="left").merge(away_stats, on="away_team", how="left", suffixes=("_home", "_away"))
m["xg_diff_delta"] = m["xg_diff_home"] - m["xg_diff_away"]
m["goal_diff_delta"] = m["goal_diff_home"] - m["goal_diff_away"]
m["win_pct_delta"] = m["win_pct_home"] - m["win_pct_away"]

X_new = m[feature_cols].fillna(0.0).values
m["home_win_prob"] = model.predict_proba(X_new)[:, 1]

matchup_output = m[["home_team", "away_team", "home_win_prob"]].sort_values("home_win_prob", ascending=False)
print("\nRound 1 matchup probabilities (home team win):")
display(matchup_output)

# --- 10) Power rankings of 32 teams (model-implied strength) ---
# Use learned weights to combine TEAM season stats into a single strength number
# This is consistent with how the model uses deltas.
# (We exclude intercept because it's constant across teams.)
coef_xg, coef_goal, coef_win = model.coef_[0]  # corresponds to deltas
team_table["model_strength"] = (
    coef_xg * team_table["xg_diff"] +
    coef_goal * team_table["goal_diff"] +
    coef_win * team_table["win_pct"]
)

power_rankings = team_table.sort_values("model_strength", ascending=False).reset_index(drop=True)
power_rankings["rank"] = power_rankings.index + 1

print("\nPower rankings (1 = strongest):")
display(power_rankings[["rank", "team", "model_strength", "xg_diff", "goal_diff", "win_pct"]])

# --- 11) Save outputs (optional) ---
# These CSVs are what you’ll paste from / upload later.
power_rankings[["rank", "team", "model_strength"]].to_csv("power_rankings_1a.csv", index=False)
matchup_output.to_csv("round1_home_win_probs_1a.csv", index=False)

print("\nSaved files:")
print(" - power_rankings_1a.csv")
print(" - round1_home_win_probs_1a.csv")